# 🎯 Interactive Substack Analyzer with LSTM

This notebook provides an interactive environment to analyze your Substack with:
- **LSTM-based predictions** with time-decay weighting
- **Real-time analysis** of your actual Substack
- **Mastery scoring** across 5 dimensions
- **Interactive visualizations**

---

## 📦 Setup & Configuration

In [ ]:
# Install dependencies (run once)
!pip install -q pandas numpy scikit-learn matplotlib plotly torch feedparser beautifulsoup4 requests nltk wordcloud

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Import our custom modules
from data_collector import SubstackCollector
from lstm_predictor import LSTMArticlePredictor
from mastery_scorer import MasteryScorer
from book_recommender import BookRecommender

print("✅ All imports successful!")

## 🔗 Connect to Your Substack

Enter your Substack URL below:

In [ ]:
# Configuration
SUBSTACK_URL = "https://substack.com/@ethannorton"  # Change this to your Substack URL
MAX_ARTICLES = 50  # Number of articles to analyze
DECAY_RATE = 0.95  # Time decay rate (0.9-0.99) - higher = more emphasis on recent

print(f"📊 Analyzing: {SUBSTACK_URL}")
print(f"📈 Max articles: {MAX_ARTICLES}")
print(f"⏱️  Decay rate: {DECAY_RATE} (emphasizing recent content)")

## 📥 Step 1: Collect Your Articles

In [ ]:
# Collect articles
collector = SubstackCollector(SUBSTACK_URL)
articles_df = collector.collect_articles(max_articles=MAX_ARTICLES)

print(f"\n✅ Collected {len(articles_df)} articles")
print(f"\nDate range: {articles_df['published_date'].min()} to {articles_df['published_date'].max()}")

# Display first few articles
articles_df[['title', 'published_date']].head(10)

## 🎯 Step 2: Calculate Mastery Score

In [ ]:
# Calculate mastery
scorer = MasteryScorer()
mastery_results = scorer.analyze_mastery(articles_df, articles_df)

# Display results
print(f"\n{'='*70}")
print(f"🎯 MASTERY ANALYSIS")
print(f"{'='*70}")
print(f"\n📊 Overall Score: {mastery_results['overall_score']:.1f}/100")
print(f"🏆 Level: {mastery_results['mastery_level']}")
print(f"\n📈 Component Scores:")

for metric, score in mastery_results['metrics'].items():
    bar = '█' * int(score / 5) + '░' * (20 - int(score / 5))
    print(f"  {metric:20s}: {bar} {score:.1f}/100")

print(f"\n💪 Strengths:")
for i, strength in enumerate(mastery_results['strengths'], 1):
    print(f"  {i}. {strength}")

print(f"\n🎯 Growth Areas:")
for i, area in enumerate(mastery_results['growth_areas'], 1):
    print(f"  {i}. {area}")

### Visualize Mastery Score

In [ ]:
# Create gauge chart
fig = go.Figure(go.Indicator(
    mode = "gauge+number+delta",
    value = mastery_results['overall_score'],
    domain = {'x': [0, 1], 'y': [0, 1]},
    title = {'text': "Overall Mastery Score", 'font': {'size': 24}},
    delta = {'reference': 70},
    gauge = {
        'axis': {'range': [None, 100]},
        'bar': {'color': "darkblue"},
        'steps': [
            {'range': [0, 40], 'color': "lightgray"},
            {'range': [40, 70], 'color': "gray"},
            {'range': [70, 100], 'color': "lightgreen"}],
        'threshold': {
            'line': {'color': "red", 'width': 4},
            'thickness': 0.75,
            'value': 90}}))

fig.update_layout(height=400)
fig.show()

# Component breakdown
metrics_df = pd.DataFrame([
    {'Metric': k.replace('_score', '').title(), 'Score': v}
    for k, v in mastery_results['metrics'].items()
])

fig = px.bar(metrics_df, x='Score', y='Metric', orientation='h',
             title='Mastery Component Breakdown',
             color='Score', color_continuous_scale='Viridis')
fig.update_layout(height=400)
fig.show()

## 🤖 Step 3: LSTM Predictions with Time-Decay Weighting

In [ ]:
# Initialize LSTM predictor
lstm_predictor = LSTMArticlePredictor(
    sequence_length=5,
    decay_rate=DECAY_RATE
)

print("Training LSTM model with time-decay weighting...")
print(f"Recent articles weighted {DECAY_RATE:.0%} more than older ones\n")

# Fit the model
lstm_predictor.fit(articles_df, train_model=True)

# Generate predictions
predictions = lstm_predictor.predict_next_articles(num_predictions=10)

print(f"\n✅ Generated {len(predictions)} predictions!")

### View Predictions

In [ ]:
# Display predictions
print(f"\n{'='*70}")
print("🔮 PREDICTED ARTICLE TOPICS (LSTM-Based)")
print(f"{'='*70}\n")

for i, pred in enumerate(predictions, 1):
    confidence_emoji = "🟢" if pred['confidence'] > 0.75 else "🟡" if pred['confidence'] > 0.5 else "🟠"
    
    print(f"{confidence_emoji} {i}. {pred['title']}")
    print(f"   Confidence: {pred['confidence']:.0%}")
    print(f"   Method: {pred['method']}")
    print(f"   Top Topics: {', '.join(pred['topics'][:5])}")
    
    if 'attention_score' in pred:
        print(f"   Attention on Recent: {pred['attention_score']:.0%}")
    
    print(f"   Description: {pred['description'][:100]}...")
    print()

### Visualize Prediction Confidence

In [ ]:
# Create prediction visualization
pred_df = pd.DataFrame([
    {'Title': pred['title'][:50], 'Confidence': pred['confidence']}
    for pred in predictions
])

fig = px.bar(pred_df, x='Confidence', y='Title', orientation='h',
             title='Prediction Confidence Scores',
             color='Confidence', color_continuous_scale='RdYlGn')
fig.update_layout(height=500, yaxis={'categoryorder':'total ascending'})
fig.show()

### Topic Trends with Time Weighting

In [ ]:
# Get topic trends
trends_df = lstm_predictor.get_topic_trends()

print("\n📈 TOPIC TRENDS (Time-Weighted)\n")
print(trends_df.to_string(index=False))

# Visualize
fig = px.scatter(trends_df, x='Topic', y='Importance',
                 size='Importance', color='Trend',
                 title='Topic Importance (Recent Articles Weighted Higher)',
                 hover_data=['Keywords'])
fig.update_layout(height=500)
fig.show()

## 📚 Step 4: Book Recommendations

In [ ]:
# Generate book recommendations
recommender = BookRecommender()
books = recommender.recommend_books(articles_df, predictions, num_books=10)

print(f"\n{'='*70}")
print("📚 PERSONALIZED BOOK RECOMMENDATIONS")
print(f"{'='*70}\n")

for i, book in enumerate(books, 1):
    stars = "⭐" * int(book['relevance_score'] * 5)
    print(f"{i}. {book['title']}")
    print(f"   by {book['author']}")
    print(f"   Relevance: {stars} ({book['relevance_score']:.0%})")
    print(f"   Topics: {', '.join(book['topics'][:3])}")
    print(f"   Why: {book['reason']}")
    print()

## 📊 Step 5: Publishing Analytics

In [ ]:
# Publishing frequency over time
articles_df['year_month'] = pd.to_datetime(articles_df['published_date']).dt.to_period('M')
freq_df = articles_df.groupby('year_month').size().reset_index(name='count')
freq_df['year_month'] = freq_df['year_month'].astype(str)

fig = px.bar(freq_df, x='year_month', y='count',
             title='Publishing Frequency Over Time',
             labels={'year_month': 'Month', 'count': 'Articles Published'},
             color='count', color_continuous_scale='Blues')
fig.update_layout(height=400)
fig.show()

# Article length distribution
articles_df['word_count'] = articles_df['content'].str.split().str.len()

fig = px.histogram(articles_df, x='word_count', nbins=20,
                   title='Article Length Distribution',
                   labels={'word_count': 'Word Count'},
                   color_discrete_sequence=['#667eea'])
fig.update_layout(height=400)
fig.show()

## 🎯 Step 6: Content Roadmap Timeline

In [ ]:
from datetime import timedelta

# Create timeline
fig = go.Figure()

# Historical articles
fig.add_trace(go.Scatter(
    x=articles_df['published_date'],
    y=[1] * len(articles_df),
    mode='markers',
    name='Published',
    marker=dict(size=12, color='blue', symbol='circle'),
    text=articles_df['title'],
    hovertemplate='<b>%{text}</b><br>%{x}<extra></extra>'
))

# Predicted articles
last_date = articles_df['published_date'].max()
pred_dates = [last_date + timedelta(days=7*(i+1)) for i in range(len(predictions))]
pred_titles = [p['title'] for p in predictions]

fig.add_trace(go.Scatter(
    x=pred_dates,
    y=[1] * len(predictions),
    mode='markers',
    name='Predicted',
    marker=dict(size=14, color='red', symbol='diamond'),
    text=pred_titles,
    hovertemplate='<b>%{text}</b><br>Suggested: %{x}<extra></extra>'
))

fig.update_layout(
    title='Content Roadmap: Past & Future',
    xaxis_title='Date',
    yaxis=dict(showticklabels=False),
    height=400,
    hovermode='closest'
)

fig.show()

## 💾 Step 7: Export Results

In [ ]:
# Save predictions
import json

output_data = {
    'mastery_score': mastery_results['overall_score'],
    'mastery_level': mastery_results['mastery_level'],
    'predictions': predictions,
    'books': books,
    'analysis_date': datetime.now().isoformat(),
    'total_articles': len(articles_df),
    'decay_rate': DECAY_RATE
}

with open('lstm_analysis_results.json', 'w') as f:
    json.dump(output_data, f, indent=2)

print("✅ Results saved to 'lstm_analysis_results.json'")
print("\n📊 Analysis Complete!")

## 🔬 Experiment: Adjust Time Decay

Try different decay rates to see how it affects predictions:

In [ ]:
# Compare different decay rates
decay_rates = [0.85, 0.90, 0.95, 0.99]
comparison_results = []

for decay in decay_rates:
    print(f"\nTesting decay rate: {decay}")
    
    predictor = LSTMArticlePredictor(decay_rate=decay)
    predictor.fit(articles_df, train_model=False)  # Skip LSTM training for speed
    preds = predictor.predict_next_articles(num_predictions=3)
    
    avg_confidence = np.mean([p['confidence'] for p in preds])
    comparison_results.append({
        'Decay Rate': decay,
        'Avg Confidence': avg_confidence,
        'Top Prediction': preds[0]['title'][:40]
    })

comparison_df = pd.DataFrame(comparison_results)
print("\n📊 Decay Rate Comparison:")
print(comparison_df.to_string(index=False))

fig = px.line(comparison_df, x='Decay Rate', y='Avg Confidence',
              title='How Decay Rate Affects Prediction Confidence',
              markers=True)
fig.show()

---

## 🎉 Summary

You've completed a full analysis of your Substack with:
- ✅ LSTM-based predictions with time-decay weighting
- ✅ Comprehensive mastery scoring
- ✅ Personalized book recommendations
- ✅ Interactive visualizations

**Next Steps:**
1. Try different SUBSTACK_URLs
2. Adjust DECAY_RATE to emphasize recent content more/less
3. Modify MAX_ARTICLES to analyze more/less history
4. Export results and track your progress over time